In [ ]:
pip install datasets

In [ ]:
from datasets import load_dataset
from huggingface_hub import notebook_login

# Log in to Hugging Face Hub (this will prompt for your token)
notebook_login()

# Load the dataset from Hugging Face
dataset = load_dataset('ainlpml/english-hindi')

# Display information about the dataset
print(dataset)

In [ ]:
import pandas as pd


df = dataset['train'].to_pandas()

split_point = len(df) // 2
# Extract English sentences
english_sentences = df['text'].iloc[:split_point].reset_index(drop=True);
# Extract Hindi sentences
hindi_sentences = df['text'].iloc[split_point:].reset_index(drop=True);
# Create a new DataFrame with paired English and Hindi texts
parallel_df = pd.DataFrame({
    'english': english_sentences,
    'hindi': hindi_sentences
});

print("\nFirst 5 entries of the parallel dataset:");
display(parallel_df.head());

print("\nLast 5 entries of the parallel dataset:");
display(parallel_df.tail());

print(f"\nShape of the parallel dataset: {parallel_df.shape}");

### Step 3: Word Count Analysis and Filtering

In [ ]:

parallel_df['word_count_english'] = parallel_df['english'].apply(lambda x: len(str(x).split()))
parallel_df['word_count_hindi'] = parallel_df['hindi'].apply(lambda x: len(str(x).split()))


filtered_df = parallel_df[
    (parallel_df['word_count_english'] >= 5) & (parallel_df['word_count_english'] <= 70) &
    (parallel_df['word_count_hindi'] >= 5) & (parallel_df['word_count_hindi'] <= 70)
].copy()

print(f"Original parallel_df shape: {parallel_df.shape}")
print(f"Filtered df shape after word count range: {filtered_df.shape}")
display(filtered_df.head())

### Step 4: Word Count Difference Calculation and Filtering

In [ ]:
filtered_df['word_count_difference'] = filtered_df['word_count_english'] - filtered_df['word_count_hindi']

final_df = filtered_df[
    (filtered_df['word_count_difference'] >= -10) & (filtered_df['word_count_difference'] <= 10)
].copy()

print(f"Filtered df shape after word count difference range: {final_df.shape}")
display(final_df.head())

### Step 5: Final Output Preparation

In [ ]:
final_df['character_count_english'] = final_df['english'].apply(len)
final_df['character_count_hindi'] = final_df['hindi'].apply(len)

# Calculate the difference in character count
final_df['character_count_difference'] = final_df['character_count_english'] - final_df['character_count_hindi']

# Prepare the cleaned dataset with the specified columns
cleaned_dataset = final_df[[
    'english',
    'hindi',
    'word_count_english',
    'word_count_hindi',
    'word_count_difference',
    'character_count_english',
    'character_count_hindi',
    'character_count_difference'
]].copy()

print("Cleaned dataset head:")
display(cleaned_dataset.head())

print(f"Cleaned dataset shape: {cleaned_dataset.shape}")

In [ ]:
# Export the results to an Excel file
output_filename = 'english_hindi_dataset_analysis.xlsx'
cleaned_dataset.to_excel(output_filename, index=False)

print(f"Dataset exported successfully to '{output_filename}'")